### HALI v2 — HPV Awareness & Learning Initiative
#### OpenAI Agents SDK | Week 2 Community Contribution

Upgrading HALI to use the OpenAI Agents SDK.

**Patterns used:**
- **Handoffs** — triage agent routes to Caregiver Agent or CHW Agent
- **Input guardrail** — blocks off-topic messages before they reach HALI

**Agents:**
- `HALI Triage` — decides who handles the message
- `Caregiver Agent` (Nurse Amina) — for parents and families, English + Swahili
- `CHW Agent` — for community health volunteers, evidence-based talking points
- `Guardrail Agent` — silently checks if the message is health-related

**Context:** Cervical cancer kills 3,400 Kenyan women every year. The vaccine is free, safe, and one dose is enough, but myths and misinformation keep uptake low in many counties.


##### The imports

In [ ]:
import os
from dotenv import load_dotenv
from agents import Agent, Runner, handoff, GuardrailFunctionOutput, input_guardrail
from agents import OpenAIResponsesModel
from pydantic import BaseModel

load_dotenv(override=True)

##### Define the specialist agents

In [ ]:
caregiver_agent = Agent(
    name="Caregiver Agent",
    instructions="""You are Nurse Amina, a warm and trusted health educator helping 
Kenyan families understand the HPV vaccine. You speak English and Swahili.
You address myths directly but gently. You never shame or lecture.
Key facts:
- The vaccine is free, safe, and one dose is enough for girls 9-14
- It does NOT cause infertility — this is a common myth
- It is halal — no pork or alcohol in the vaccine
- It prevents cervical cancer, which kills 3,400 Kenyan women every year
If a parent wants to register interest, ask for their name and county.""",
)

chv_agent = Agent(
    name="CHV Agent", 
    instructions="""You are a clinical support assistant for Community Health Volunteers 
in Kenya. Give evidence-based talking points, handle objections with data.
Be concise and practical — CHVs need answers they can use in the field immediately.
Cite WHO or Kenya MOH guidelines where relevant.""",
)


##### Triage agent with handoffs

In [ ]:
triage_agent = Agent(
    name="HALI Triage",
    instructions="""You are HALI — Health & Wellbeing, an HPV vaccine companion for Kenya.
Decide who should handle the message:
- If the user is a parent, caregiver, or family member → hand off to Caregiver Agent
- If the user is a nurse, CHV, or health worker → hand off to CHV Agent
- If unclear, ask one short question to find out.""",
    handoffs=[caregiver_agent, chv_agent],
)

##### Add guardrails

In [ ]:
class GuardrailOutput(BaseModel):
    is_hpv_related: bool
    reasoning: str

guardrail_agent = Agent(
    name="Guardrail",
    instructions="""Decide if the user's message is related to HPV, cervical cancer, 
vaccines, or general health questions relevant to Kenya. 
Return is_hpv_related=True if it is, False if it is completely off-topic (e.g. cooking, sport, politics).""",
    output_type=GuardrailOutput,
)

@input_guardrail
async def hpv_guardrail(ctx, agent, message):
    result = await Runner.run(guardrail_agent, message, context=ctx.context)
    output = result.final_output
    return GuardrailFunctionOutput(
        output_info=output,
        tripwire_triggered=not output.is_hpv_related,
    )

triage_agent.input_guardrails = [hpv_guardrail]

Test triage handoff to Caregiver Agent.

In [ ]:
result = await Runner.run(triage_agent, "I'm a mother in Kisumu. I heard the vaccine causes infertility — is that true?")
print(result.final_output)

Test guardrails

In [ ]:
from agents import InputGuardrailTripwireTriggered

try:
    result = await Runner.run(triage_agent, "Who won the Champions League last night?")
    print(result.final_output)
except InputGuardrailTripwireTriggered:
    print("Guardrail triggered — off-topic message blocked.")


Test the CHV handoff

In [ ]:
result = await Runner.run(triage_agent, "I'm a community health volunteer in Wajir. A mother says the vaccine is haram — what do I tell her?")
print(result.final_output)